In [1]:
import os
import numpy as np
import pandas as pd
from scipy.ndimage import label

columns = [
    'Object','Index','nr_pix','rows_with_1','cols_with_1',
    'rows_with_2','cols_with_2','rows_with_3p','cols_with_3p',
    'height', 'width', 'aspect_ratio', 'maxrow', 'maxcol',
    'connected_areas', 'eyes', 'hollowness', 'custom'
]

feature_data = []

<h1>Initialising data collection</h1>

In [2]:

student_ID = "40443014"

#Object labels (from section 1)
living_objects = ['banana', 'cherry', 'lemon', 'tree']
non_living_objects = ['envelope','golfclub','pencil','wineglass']
all_objects = living_objects + non_living_objects

for object in all_objects:
    for i in range(1,15):
        index_str = f"{i:02d}"
        file_path = f"./images/{student_ID}_{object}_{index_str}.csv" 

        if os.path.exists(file_path):
            matrix = np.genfromtxt(file_path, delimiter=',')

            #Basic Sums
            nr_pix = np.sum(matrix)
            row_sums = np.sum(matrix, axis = 1)
            col_sums = np.sum(matrix, axis = 0)

            #Pixel counts
            # 1 black sqrs
            rows_with_1 = np.sum(row_sums == 1)
            cols_with_1 = np.sum(col_sums == 1)

            # 2 black sqrs
            rows_with_2 = np.sum(row_sums == 2)
            cols_with_2 = np.sum(col_sums == 2)

            #3 black sqrs
            rows_with_3p = np.sum(row_sums > 2)
            cols_with_3p = np.sum(col_sums > 2)

            #Image Dimensions
            row_index, col_index = np.where(matrix == 1)

            if len(row_index) > 0:
                height = np.max(row_index) - np.min(row_index)
                width = np.max(col_index) - np.min(col_index)
                aspect_ratio = width / height if height >0 else 0
            else:
                height, width, aspect_ratio = 0, 0, 0

            #Max row & col
            maxrow = np.max(row_sums)
            maxcol = np.max(col_sums)

            #connected areas
            block_grid = np.ones((3, 3))
            _, connected_areas = label(matrix, structure=block_grid)

            #Eyes
            invert_matrix = np.logical_not(matrix).astype(int)
            white_map, num_white_areas = label(invert_matrix)

            eyes = num_white_areas - 1

            #Hollowness
            if nr_pix > 0:
                sum_centre_white_pixels = np.sum(white_map > 1)
                hollowness = sum_centre_white_pixels / nr_pix
            else:
                hollowness = 0

            #Custom (Extent)
            if(height * width) > 0:
                custom = nr_pix / (height * width)
            else:
                custom = 0

            #Hold all information
            information_row = [
                object,index_str,nr_pix,rows_with_1,cols_with_1,
                rows_with_2,cols_with_2,rows_with_3p,cols_with_3p,
                height, width, aspect_ratio, maxrow, maxcol,
                connected_areas, int(eyes), hollowness, custom
            ]

            feature_data.append(information_row)

<h1>Calculating Feature</h1>
Firstly, save all objects and create a for loop that takes every object and goes through all 14 csv files in each object to collect the necessary information on the object.
For loop first checks if file exists and finds it, then split all blocks with a ','.

<h3>-Basic Sums-</h3>
Just adds all numbers together (as black is 1 and white is 0).
row and col sum use axis to collect the number of black squares in all the individual rows and col (use this for Pixel counts)

<h3>-Pixel Counts-</h3>
All use the same format of taking the information from the row and col sums and checking to see if black square == 1, 2 or are > 2.

<h3>-Image Dimensions-</h3>
First, find co-ordinates of all blocks that contain a 1.
Then using this to find height, take the smallest row that contains a 1 and subtract it from the largest row containing a 1.
To find width just do the same but take the smallest collumn with a 1 away from the largest collumn with a 1.
Aspect ratio is just width /height (as outlined by the requirements). To avoid any errors exception handling  was added (if height = 0, so no ZeroDivision error for division using a 0)

<h3>-Max row and collumn-</h3>
Just takes the row and collumn with the most black squares from row and col sums using the 'max' function

<h3>-Connectd Areas-</h3>
This part utilised the scipy.ndimage.label function.
Firstly, created a 3 by 3 grid filled with 1's using the numpy.one function. This would act as the example that the label function would be searching for (As for a connected area even diagonal squares are counted). The label scans the matrix and finds a black square which it will label (like blk_sqr_01 for example). Using the 3x3 grid the label scans out all around the black squareto see if there are anymore 1's. If there is, it will move to that square, label it the same as the previous square and continue the search until it finds a dead end. When this happens, the label function will continue scanning the matrix until it finds another unlabeled black square and repeats the scanning or until it gets to the last block in the matrix and there are no more black squares to count. When this is completed the label function returns two variables, the first being a new matrix where every pixel is now labled (like blk_sqr_01, blk_sqr_02, ect) and the second being a count of the objects found. Since in this case only the count of objects is needed, the new matrix is disregarded (using'_') and the object count is collected in the 'connected_area' variable.

<h3>-Eyes-</h3>
Eyes are essentially the same as connected areas except that the connected areas are white and not black. So the same logic is applied. To allow this the matrix has to first be inverted (all '1' = '0' and '0' = '1'), so that code from above can be reused. As all white squares are now 'black' the code can recognise and count them. Technically speaking the label function only returns all objects and not specifically regions that are surreounded by lines however using logic it can be deduced that all images will contain at least one object (that being the white around the drawing) so anymore objects after that will be eyes surrounded by lines. Using this logic, the total sum of eyes will equal all objects found subtracted by 1 (The surrounding white)

<h3>-Hollowness-</h3>
Firstly, the grid and labels that the eyes label created (this is stored in 'white_map') are taken. As this feature required division some error handling (in the form of making sure there were black pixels to divide by to avoid any ZeroDivisionErrors) was required to begin. From the Eyes label all of the white_map matrix regions have been labeled as 1,2,3 with the label 1 being the white surrounding the picture. To avoid these white pixels being counted, "white_map > 1" is implemented into the sum. Any white squares now counted are only ones inside the drawings (Eyes). Then following the specification outlined, the sum of white squares in the eyes are divided by the sum of all black squares.

<h3>-Custom-</h3>
In the custom feature we want to find the ectent of an object (Basically how much of the objects "bounding box" is taken up by black pixels) To get this we get the objects total black pixels and the objects box (Height * Width) and divide the total by the box. This will give us the extent.

<h3>------------------------------------------------------------------</h3>
<p>All of this information is then stored in 'information_row' and saved to 'feature_data'. This will then hold all data collected within all objects. </p>

In [3]:
data_frame = pd.DataFrame(feature_data, columns=columns)

output_filename = f"{student_ID}_features.csv"

data_frame.to_csv(output_filename, index = False)

print(f"{output_filename} has successfully been created")
data_frame.head()

40443014_features.csv has successfully been created


,Object,Index,nr_pix,rows_with_1,cols_with_1,rows_with_2,cols_with_2,rows_with_3p,cols_with_3p,height,width,aspect_ratio,maxrow,maxcol,connected_areas,eyes,hollowness,custom
0,banana,01,97.0,0,0,5,7,20,19,24,25,1.041667,7.0,11.0,1,1,1.402062,0.161667
1,banana,02,101.0,0,0,7,8,19,20,25,27,1.080000,11.0,10.0,1,1,1.801980,0.149630
2,banana,03,98.0,0,1,1,14,20,15,20,29,1.450000,9.0,8.0,1,1,1.285714,0.168966
3,banana,04,97.0,1,0,2,15,18,17,20,31,1.550000,12.0,7.0,1,1,1.835052,0.156452
4,banana,05,83.0,1,1,0,14,17,14,17,28,1.647059,9.0,9.0,1,1,1.686747,0.174370


<h1>Creating and storing features</h1>

1. Data_frame turns our list into a structured table. 

2. Converted this structured table into a csv file under the specified name (STUDENTN_features.csv) and set index to false as to not add a column describing row numbers.

3. Lastly a success line is printed to make sure code has ran and first 5 rows of data are printed to verify it has worked.

In [4]:
#Testing Block 

print(f"Data Dimensions: {data_frame.shape}")

if data_frame.isnull().values.any():
    print("Values are missing")
else:
    print("All values have been found")

test_columns_1 = ['Object','Index','nr_pix','rows_with_1','cols_with_1','rows_with_2']
test_columns_2 = ['cols_with_2','rows_with_3p','cols_with_3p','height', 'width', 'aspect_ratio'] 
test_columns_3 = ['maxrow', 'maxcol','connected_areas', 'eyes', 'hollowness', 'custom']

wineglass_data = data_frame[data_frame['Object'] == 'wineglass']

print("\n---* Group 1 *---")
print(wineglass_data[test_columns_1])

print("\n---* Group 2 *---")
print(wineglass_data[test_columns_2])

print("\n---* Group 3 *---")
print(wineglass_data[test_columns_3])

Data Dimensions: (112, 18)
All values have been found

---* Group 1 *---
        Object Index  nr_pix  rows_with_1  cols_with_1  rows_with_2
98   wineglass    01    77.0           11            3            9
99   wineglass    02    97.0           10            2           14
100  wineglass    03    93.0            8            1           18
101  wineglass    04    89.0           10            1           15
102  wineglass    05    86.0            8            0           19
103  wineglass    06   106.0           10            6           20
104  wineglass    07    89.0            8            0           20
105  wineglass    08    91.0            7            1           22
106  wineglass    09    73.0            4            0           17
107  wineglass    10    88.0            9            0           19
108  wineglass    11    85.0            6            1           22
109  wineglass    12    76.0            6            6           19
110  wineglass    13    84.0            9  